# Notebook 02 — Pipeline de limpieza y validación

**Proyecto:** Auditoría de desempeño y costo en una red de transporte terrestre
**Autor:** Sergio Corona

## Objetivo

Convertir las 8 reglas de limpieza derivadas del diagnóstico (notebook 01) en una
clase reutilizable (`src/limpieza.py`), aplicarla al archivo crudo y validar el
resultado contra la versión limpia oficial del dataset.

## Criterio de éxito

El dataset producido por el pipeline debe coincidir con `fact_shipments_clean.csv`
en número de filas, unicidad de identificadores y distribución de las columnas
críticas. Las divergencias que existan deben quedar explicadas, no ocultadas.

## Insumos

| Archivo | Rol |
|---|---|
| `data/raw/fact_shipments_raw.csv` | Entrada del pipeline |
| `data/raw/fact_shipments_clean.csv` | Referencia de validación |

> El archivo limpio se usa **únicamente** para validar al final. Ninguna regla del
> pipeline se deriva de él: todas provienen del diagnóstico del notebook 01.

In [1]:
# Recarga automática: al guardar src/limpieza.py, los cambios se reflejan
# en el notebook sin reiniciar el kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Permite importar desde src/ usando ruta relativa al notebook
sys.path.append(str(Path.cwd().parent / "src"))

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)
print("src en sys.path:", str(Path.cwd().parent / "src"))

pandas: 3.0.3
numpy : 2.5.1
src en sys.path: C:\Users\Sergio Corona\Desktop\transportation-analytics\src


## 1. Carga de datos

Se cargan dos archivos con roles distintos:

- **`fact_shipments_raw.csv`** — entrada del pipeline. Contiene los problemas de calidad
  diagnosticados en el notebook 01.
- **`fact_shipments_clean.csv`** — referencia de validación. Se carga aquí para tenerlo
  disponible, pero **no se consulta hasta la sección 4**.

Las fechas se cargan como texto de forma deliberada: la conversión de tipos es el paso 3
del pipeline y debe ocurrir dentro de la clase, no en la lectura del archivo. Si se
convirtieran aquí, el pipeline no sería reproducible desde el archivo crudo.

In [2]:
RUTA_RAW = Path.cwd().parent / "data" / "raw"

df_raw = pd.read_csv(RUTA_RAW / "fact_shipments_raw.csv")
df_clean_ref = pd.read_csv(RUTA_RAW / "fact_shipments_clean.csv")

print(f"raw   : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"clean : {df_clean_ref.shape[0]:,} filas × {df_clean_ref.shape[1]} columnas")
print(f"\nDiferencia de filas: {df_raw.shape[0] - df_clean_ref.shape[0]:,}")
print(f"¿Mismas columnas?   : {list(df_raw.columns) == list(df_clean_ref.columns)}")

raw   : 221,320 filas × 39 columnas
clean : 220,000 filas × 39 columnas

Diferencia de filas: 1,320
¿Mismas columnas?   : True


**Hallazgo.** Ambos archivos comparten las mismas 39 columnas: la versión oficial limpia
no incorpora columnas derivadas. Las métricas del paso 8 (OTD, costo por kg, costo por km)
son producto de este pipeline y no tienen contraparte en la referencia; se validarán por
consistencia interna, no por comparación.

La diferencia de 1,320 filas coincide con el número de identificadores duplicados
detectado en el notebook 01, lo que confirma que el tratamiento oficial de duplicados
consiste en conservar un solo registro por `shipment_id`.

In [3]:
from limpieza import (
    LimpiadorEmbarques,
    MAPA_MODO,
    MAPA_PAIS,
    COLUMNAS_FECHA,
    TOLERANCIA_OTD_HORAS,
)

print("Import correcto")
print(f"Modos mapeados : {len(MAPA_MODO)} claves → {sorted(set(MAPA_MODO.values()))}")
print(f"Países mapeados: {len(MAPA_PAIS)} claves → {sorted(set(MAPA_PAIS.values()))}")
print(f"Tolerancia OTD : {TOLERANCIA_OTD_HORAS} horas")
print(f"\nColumnas de fecha declaradas vs. presentes en el raw:")
for col in COLUMNAS_FECHA:
    print(f"  {col:24} {'OK' if col in df_raw.columns else 'NO EXISTE'}")

Import correcto
Modos mapeados : 8 claves → ['Air', 'Drayage', 'LTL', 'Ocean', 'Parcel', 'TL']
Países mapeados: 6 claves → ['CA', 'MX', 'US']
Tolerancia OTD : 12 horas

Columnas de fecha declaradas vs. presentes en el raw:
  ship_date                OK
  planned_pickup_ts        OK
  planned_delivery_ts      OK
  actual_pickup_ts         OK
  actual_delivery_ts       OK


In [4]:
limpiador = LimpiadorEmbarques(df_raw)
limpiador.normalizar_categoricas()

print("\n--- Verificación ---")
for col in ["mode", "origin_country", "dest_country"]:
    print(f"\n{col}: {limpiador.df[col].nunique()} valores únicos")
    print(limpiador.df[col].value_counts().to_string())

print(f"\n¿df_raw sigue intacto? {df_raw['mode'].nunique()} valores únicos (esperado: 14)")

[1-normalizar] mode: 6,698
[1-normalizar] origin_country: 4,426
[1-normalizar] dest_country: 0

--- Verificación ---

mode: 6 valores únicos
mode
Parcel     84709
LTL        53014
TL         41922
Ocean      17528
Air        15333
Drayage     8814

origin_country: 3 valores únicos
origin_country
US    116761
MX     85678
CA     18881

dest_country: 3 valores únicos
dest_country
US    111820
MX     72646
CA     36854

¿df_raw sigue intacto? 14 valores únicos (esperado: 14)


### Paso 1 — Normalizar categóricas

**Resultado:** 11,124 valores corregidos (6,698 en `mode`, 4,426 en `origin_country`,
0 en `dest_country`), coincidiendo exactamente con las cifras del diagnóstico.

La consolidación por modo es aritméticamente exacta: cada categoría final equivale a la
suma de sus variantes de origen. Ningún registro se perdió ni se reasignó a un modo
incorrecto.

`dest_country` reporta 0 modificaciones, confirmando el hallazgo del notebook 01: el
problema de formato no era simétrico entre columnas gemelas.

El DataFrame de entrada conserva sus 14 variantes de `mode`, verificando que la copia
defensiva del constructor aísla el pipeline del objeto original.

In [5]:
limpiador.deduplicar()

print("\n--- Verificación ---")
print(f"Filas             : {len(limpiador.df):,} (esperado: 220,000)")
print(f"IDs únicos        : {limpiador.df['shipment_id'].is_unique}")
print(f"Costos negativos  : {(limpiador.df['total_cost_usd'] < 0).sum()} (esperado: 393)")
print(f"¿Sobra _valido?   : {'_valido' in limpiador.df.columns} (esperado: False)")

[2-deduplicar] registros eliminados: 1,320

--- Verificación ---
Filas             : 220,000 (esperado: 220,000)
IDs únicos        : True
Costos negativos  : 393 (esperado: 393)
¿Sobra _valido?   : False (esperado: False)


### Paso 2 — Deduplicar

**Resultado:** 1,320 registros eliminados. El dataset queda en 220,000 filas con
`shipment_id` único.

El criterio de puntaje de validez se verifica por sus efectos: quedan 393 costos
negativos, no 400. Los 7 identificadores duplicados donde una copia tenía el signo
invertido se resolvieron conservando la copia sana; los 393 restantes son casos sin
gemelo, que corresponden al paso 5.

Una deduplicación por `keep='first'` habría producido 220,000 filas igualmente, pero con
una composición distinta: la elección entre copias habría dependido del orden de
aparición en el archivo. El conteo de filas por sí solo no distingue ambos resultados.

El ordenamiento usa `kind="stable"` para que los empates de puntaje —los 1,186
duplicados exactos, donde ambas copias son idénticas— conserven el orden original y el
pipeline sea reproducible entre

In [6]:
limpiador.convertir_fechas()

print("\n--- Verificación ---")
print(limpiador.df[COLUMNAS_FECHA].dtypes.to_string())

print("\nRangos:")
for col in COLUMNAS_FECHA:
    print(f"  {col:22} {limpiador.df[col].min()}  →  {limpiador.df[col].max()}")

print("\nNulos:")
print(limpiador.df[COLUMNAS_FECHA].isna().sum().to_string())

viol = (limpiador.df["actual_delivery_ts"] < limpiador.df["actual_pickup_ts"]).sum()
print(f"\nEntregas anteriores al pickup: {viol} (esperado: 298)")

[3-fechas] ship_date → datetime: 220,000
[3-fechas] planned_pickup_ts → datetime: 220,000
[3-fechas] planned_delivery_ts → datetime: 220,000
[3-fechas] actual_pickup_ts → datetime: 220,000
[3-fechas] actual_delivery_ts → datetime: 220,000

--- Verificación ---
ship_date              datetime64[us]
planned_pickup_ts      datetime64[us]
planned_delivery_ts    datetime64[us]
actual_pickup_ts       datetime64[us]
actual_delivery_ts     datetime64[us]

Rangos:
  ship_date              2023-01-01 00:00:00  →  2026-06-30 00:00:00
  planned_pickup_ts      2023-01-01 00:00:00  →  2026-06-30 00:00:00
  planned_delivery_ts    2023-01-03 00:00:00  →  2026-08-08 00:00:00
  actual_pickup_ts       2023-01-01 00:00:00  →  2026-06-30 13:24:00
  actual_delivery_ts     2023-01-01 12:00:00  →  2026-08-09 03:50:24

Nulos:
ship_date                 0
planned_pickup_ts         0
planned_delivery_ts       0
actual_pickup_ts          0
actual_delivery_ts     1756

Entregas anteriores al pickup: 298 (esperado: 

### Paso 3 — Convertir tipos temporales

**Resultado:** las 5 columnas convertidas a `datetime64[us]` sin valores no parseables.

Los rangos son coherentes con lo documentado: `ship_date` cubre 2023-01-01 a 2026-06-30.
`actual_delivery_ts` se extiende hasta 2026-08-09, lo cual es plausible: un embarque
Ocean despachado a fines de junio entrega alrededor de 40 días después.

Los nulos de `actual_delivery_ts` bajan de 1,767 a 1,756. La diferencia corresponde a
once registros eliminados en la deduplicación; el conteo del diagnóstico se hizo sobre el
archivo con duplicados.

**Nota sobre resolución temporal.** Pandas 3 infiere la unidad a partir de los valores:
asignaba `[us]` a las columnas con horas redondas y `[ns]` a `actual_delivery_ts`, la
única con segundos reales. Se fija la resolución explícitamente para que las cinco
columnas compartan dtype, evitando inconsistencias en la exportación a CSV y en la carga
a Power BI.

Se confirman las 298 violaciones de secuencia que atiende el paso 4.

In [7]:
# Muestra de control: guardamos algunos casos antes de tocarlos
mask_antes = limpiador.df["actual_delivery_ts"] < limpiador.df["actual_pickup_ts"]
muestra = limpiador.df.loc[
    mask_antes, ["shipment_id", "actual_pickup_ts", "actual_delivery_ts", "actual_transit_days"]
].head(3).copy()

limpiador.reconstruir_fechas_imposibles()

print("\n--- Antes ---")
print(muestra.to_string(index=False))

print("\n--- Después ---")
print(
    limpiador.df.loc[
        muestra.index,
        ["shipment_id", "actual_pickup_ts", "actual_delivery_ts", "actual_transit_days"],
    ].to_string(index=False)
)

viol = (limpiador.df["actual_delivery_ts"] < limpiador.df["actual_pickup_ts"]).sum()
print(f"\nViolaciones restantes: {viol} (esperado: 0)")

[4-fechas-imposibles] entregas reconstruidas: 298

--- Antes ---
shipment_id actual_pickup_ts actual_delivery_ts  actual_transit_days
 SHP1144082       2023-04-20         2023-04-18                 3.37
 SHP1139470       2023-02-09         2023-02-07                 8.82
 SHP1203370       2024-11-15         2024-11-13                 2.51

--- Después ---
shipment_id actual_pickup_ts  actual_delivery_ts  actual_transit_days
 SHP1144082       2023-04-20 2023-04-23 08:52:48                 3.37
 SHP1139470       2023-02-09 2023-02-17 19:40:48                 8.82
 SHP1203370       2024-11-15 2024-11-17 12:14:24                 2.51

Violaciones restantes: 0 (esperado: 0)


### Paso 4 — Reconstruir fechas imposibles

**Resultado:** 298 entregas recalculadas. No quedan registros con entrega anterior al
pickup.

La muestra de control confirma el patrón documentado: en los tres casos la entrega
figuraba exactamente 48 horas antes del pickup. Tras la reconstrucción, cada una cae a la
distancia que indica `actual_transit_days`, con precisión de horas y minutos —
SHP1144082 con 3.37 días de tránsito entrega a las 08:52:48 del tercer día. Esa
granularidad verifica que el cálculo usa el valor real de la columna y no un redondeo.

**Nota sobre resolución temporal.** La suma de un `datetime64[us]` con un `timedelta`
producido por `pd.to_timedelta` genera un resultado en nanosegundos, que pandas 3 rechaza
al escribirlo en la columna de destino. El resultado se convierte explícitamente antes de
asignarlo. Es consecuencia directa de haber unificado la resolución en el paso 3: la
consistencia de tipos exige mantenerla en cada operación que produzca fechas nuevas.

In [8]:
# Muestra de control
mask_neg = limpiador.df["total_cost_usd"] < 0
cols = ["shipment_id", "total_cost_usd", "linehaul_cost_usd", "fuel_surcharge_usd",
        "accessorial_usd", "customs_fee_usd", "detention_usd"]
muestra = limpiador.df.loc[mask_neg, cols].head(3).copy()

limpiador.corregir_costos_negativos()

print("\n--- Antes ---")
print(muestra.to_string(index=False))

print("\n--- Después ---")
print(limpiador.df.loc[muestra.index, cols].to_string(index=False))

print(f"\nNegativos restantes: {(limpiador.df['total_cost_usd'] < 0).sum()} (esperado: 0)")

[5-costos] signos invertidos: 393
[5-costos] validados contra sus componentes: 387
[5-costos] no validables por componente nulo: 6
[5-costos] corregidos que no cuadran: 0

--- Antes ---
shipment_id  total_cost_usd  linehaul_cost_usd  fuel_surcharge_usd  accessorial_usd  customs_fee_usd  detention_usd
 SHP1209043        -2399.93            1723.36              370.18             0.00           306.39            0.0
 SHP1063123        -3978.24            3490.68              260.75           134.09            92.72            0.0
 SHP1196963         -105.86              74.10               11.92            19.84             0.00            0.0

--- Después ---
shipment_id  total_cost_usd  linehaul_cost_usd  fuel_surcharge_usd  accessorial_usd  customs_fee_usd  detention_usd
 SHP1209043         2399.93            1723.36              370.18             0.00           306.39            0.0
 SHP1063123         3978.24            3490.68              260.75           134.09            92.72 

La validación revisa los cinco componentes, no solo `accessorial_usd`: de ahí que reporte
6 registros no validables frente a los 3 detectados en el diagnóstico inicial. Los 3
adicionales tienen nulo en alguna otra columna de costo.

In [9]:
limpiador.corregir_pesos_inflados()

d = limpiador._detalle_pesos
print("\n--- Efecto sobre los corregidos ---")
print(f"Mediana : {d['mediana_antes']:>12,.1f}  →  {d['mediana_despues']:>10,.1f} kg")
print(f"Máximo  : {d['maximo_antes']:>12,.1f}  →  {d['maximo_despues']:>10,.1f} kg")

print("\n--- Perfil de peso por modo tras la corrección ---")
print(
    limpiador.df.groupby("mode")["weight_kg"]
    .agg(["count", "median", "max"])
    .round(1)
    .to_string()
)

[6-pesos] corregidos por división entre 1000: 242
[6-pesos]   Parcel: 89
[6-pesos]   LTL: 69
[6-pesos]   TL: 45
[6-pesos]   Ocean: 18
[6-pesos]   Air: 13
[6-pesos]   Drayage: 8

--- Efecto sobre los corregidos ---
Mediana :    483,900.0  →       483.9 kg
Máximo  : 26,000,000.0  →    26,000.0 kg

--- Perfil de peso por modo tras la corrección ---
         count   median        max
mode                              
Air      15080    219.8     3000.0
Drayage   8644  13273.0    22000.0
LTL      52064    733.1     6000.0
Ocean    17233  14680.3    26000.0
Parcel   83173     11.0      900.0
TL       41166  10949.7  4000000.0


In [10]:
tl = limpiador.df[limpiador.df["mode"] == "TL"]
lim = tl["weight_kg"].quantile([0.01, 0.99])
print(f"TL — p01: {lim.iloc[0]:,.1f} kg   p99: {lim.iloc[1]:,.1f} kg\n")

print("Los 5 TL más pesados tras la corrección:")
print(
    tl.nlargest(5, "weight_kg")[["shipment_id", "weight_kg", "total_cost_usd", "distance_km"]]
    .assign(entre_1000=lambda d: d["weight_kg"] / 1000)
    .to_string(index=False)
)

TL — p01: 4,884.8 kg   p99: 21,000.0 kg

Los 5 TL más pesados tras la corrección:
shipment_id  weight_kg  total_cost_usd  distance_km  entre_1000
 SHP1103930  4000000.0         1743.91       2292.5      4000.0
 SHP1196426    21000.0         6708.05       1704.2        21.0
 SHP1002827    21000.0         9483.71       3553.1        21.0
 SHP1009900    21000.0         7063.98       1950.1        21.0
 SHP1044413    21000.0         8671.94       2607.9        21.0


### Paso 6 — Corregir pesos inflados

**Resultado:** 242 registros corregidos, con distribución por modo idéntica a la del
diagnóstico: Parcel 89 · LTL 69 · TL 45 · Ocean 18 · Air 13 · Drayage 8. La mediana de
los valores afectados pasa de 483,900 a 483.9 kg; el máximo, de 26,000,000 a 26,000 kg.

Los 89 casos de Parcel son la justificación del método. Un umbral global construido sobre
la mediana general (45,080 kg) no los detectaría: un paquete de 3 kg inflado a 3,000 kg
es absurdo para paquetería, pero queda muy por debajo de un límite calibrado con
embarques marítimos y de carga completa.

**Sobre la cobertura del criterio.** El README del dataset estima cerca de 250 casos y se
corrigen 242. El registro no cubierto más visible es SHP1103930, un TL que conserva
4,000,000 kg tras la ejecución. El criterio se abstiene deliberadamente: dividido entre
1000 daría 4,000 kg, por debajo del percentil 1 del modo (4,884.8 kg). La hipótesis del
error de unidad por factor 1000 no lo explica.

El costo respalda la abstención. El registro reporta 1,743.91 USD por 2,292 km, mientras
que los demás TL de distancia comparable cobran entre 7,000 y 9,500 USD. Ese costo es
consistente con un embarque ligero, no con uno de 4,000 kg, lo que sugiere un factor de
corrupción distinto — posiblemente 10,000 — que no puede determinarse con la evidencia
disponible.

Corregirlo exigiría suponer una transformación no demostrada. Se documenta en lugar de
ajustarse el criterio para alcanzar la cifra esperada.

In [11]:
limpiador.tratar_nulos()

print(f"\nFilas tras el paso 7: {len(limpiador.df):,}")
print(f"Diferencia vs. referencia oficial: {len(limpiador.df) - 220_000:,}")

print("\n--- Nulos restantes ---")
nulos = limpiador.df.isna().sum()
print(nulos[nulos > 0].to_string() if (nulos > 0).any() else "ninguno")

print("\n--- Incoterm tras el tratamiento ---")
print(limpiador.df["incoterm"].value_counts().to_string())

[7-nulos] carrier_id ausente → registro descartado: 880
[7-nulos] incoterm ausente → NO_APLICA: 6,569
[7-nulos] accessorial_usd ausente → 0: 3,290
[7-nulos] weight_kg ausente → mediana de su modo: 2,622
[7-nulos] volume_cbm ausente → mediana de su modo: 4,389
[7-nulos] actual_delivery_ts ausente → se conserva nulo: 1,743

Filas tras el paso 7: 219,120
Diferencia vs. referencia oficial: -880

--- Nulos restantes ---
actual_delivery_ts    1743

--- Incoterm tras el tratamiento ---
incoterm
DAP          89203
FCA          43924
EXW          27633
DDP          25773
FOB          13029
CIF          12989
NO_APLICA     6569


### Paso 7 — Tratar nulos

**Resultado:** 219,120 filas. Solo `actual_delivery_ts` conserva nulos (1,743), por
diseño.

| Columna | Tratamiento | Afectados |
|---|---|---|
| `carrier_id` | Registro descartado | 880 |
| `incoterm` | Etiqueta `NO_APLICA` | 6,569 |
| `accessorial_usd` | Imputar 0 | 3,290 |
| `weight_kg` | Mediana de su modo | 2,622 |
| `volume_cbm` | Mediana de su modo | 4,389 |
| `actual_delivery_ts` | Se conserva nulo | 1,743 |

Las cifras quedan por debajo de las del diagnóstico (887, 6,640, 3,318, 2,648, 4,422,
1,767) porque aquel conteo se hizo sobre el archivo con duplicados. La deduplicación
eliminó registros que también presentaban valores ausentes.

**Divergencia deliberada con la referencia oficial.** El resultado tiene 880 filas menos
que `fact_shipments_clean.csv`, que conserva los registros sin transportista. Es una
decisión analítica, no un error: el scorecard de carriers es el eje del proyecto, y un
embarque sin `carrier_id` no puede atribuirse a ninguno. Mantenerlo inflaría los
denominadores de volumen y gasto sin aportar al análisis.

La imputación por mediana se ejecuta después de la corrección de pesos (paso 6). El orden
importa: calcular la mediana de `weight_kg` con los 242 valores inflados aún presentes la
desplazaría hacia arriba, propagando el error a los 2,622 registros imputados.

`NO_APLICA` queda como séptima categoría de incoterm con 6,569 casos, por debajo de las
seis reales. La ausencia permanece visible como información —el incoterm no aplica en
tráfico doméstico— en lugar de desaparecer como vacío.

In [12]:
limpiador.derivar_metricas()

otd = limpiador.df["otd"].mean()
print(f"\nOTD con tolerancia de {TOLERANCIA_OTD_HORAS}h : {otd:.2%}")

# Contraste: qué pasaría sin la tolerancia
sin_tol = (
    limpiador.df["actual_delivery_ts"] <= limpiador.df["planned_delivery_ts"]
).where(limpiador.df["actual_delivery_ts"].notna())
print(f"OTD por comparación directa   : {sin_tol.mean():.2%}")
print(f"Diferencia                    : {(otd - sin_tol.mean()):.2%} puntos")

print("\n--- Métricas derivadas ---")
print(
    limpiador.df[["retraso_horas", "costo_por_kg", "costo_por_km"]]
    .describe()
    .round(2)
    .to_string()
)

print("\n--- OTD por modo ---")
print(limpiador.df.groupby("mode")["otd"].agg(["mean", "count"]).round(4).to_string())

[8-metricas] columnas derivadas: 4
[8-metricas] entregas con OTD calculable: 217,377
[8-metricas] OTD indeterminado (sin entrega): 1,743

OTD con tolerancia de 12h : 83.93%
OTD por comparación directa   : 73.80%
Diferencia                    : 10.12% puntos

--- Métricas derivadas ---
       retraso_horas  costo_por_kg  costo_por_km
count      217377.00     219120.00     219120.00
mean          -13.82         13.75          0.81
std            28.63         29.95          1.28
min           -85.44          0.00          0.00
25%           -34.10          0.47          0.14
50%           -18.44          2.19          0.43
75%             1.26         15.84          1.18
max           227.52       1682.56         76.77

--- OTD por modo ---
           mean  count
mode                  
Air      0.8721  15062
Drayage  0.8105   8637
LTL      0.8436  52089
Ocean    0.6671  17216
Parcel   0.8649  83209
TL       0.8479  41164


    ### Paso 8 — Derivar métricas

**Resultado:** cuatro columnas nuevas (`otd`, `retraso_horas`, `costo_por_kg`,
`costo_por_km`) sobre 219,120 registros. El OTD se calcula sobre 217,377 entregas
confirmadas; 1,743 quedan indeterminados por falta de fecha de entrega.

**Validación de la regla de tolerancia.** El OTD resultante es 83.93%, frente al 83.92%
que reporta el flag original del sistema. La diferencia de una centésima corresponde a
los 880 registros descartados en el paso 7. La regla derivada de auditar 25 casos
discrepantes se sostiene sobre el dataset completo.

El contraste cuantifica lo que estaba en juego: la comparación directa de timestamps
arroja 73.80%, **10.12 puntos por debajo** del valor oficial. Un tablero construido sobre
ese supuesto contradiría los reportes del área de transporte sin que ninguna de las dos
cifras fuera un error de cálculo.

**Los no entregados se marcan como indeterminados, no como incumplimientos.** La columna
usa el dtype `boolean` de pandas, que admite nulo. Asignarles `False` los contaría como
entregas fuera de plazo y deprimiría el indicador; un embarque sin fecha de entrega puede
estar en tránsito. El denominador queda declarado: entregas confirmadas.

**Hallazgos preliminares para la fase de análisis.**

El retraso mediano es de −18.44 horas y el percentil 75 apenas alcanza +1.26 horas: tres
de cada cuatro embarques llegan antes del compromiso. Esto reencuadra la tolerancia de
12 horas — no compensa un desempeño deficiente generalizado, sino que delimita la cola de
la distribución.

El OTD por modo muestra un escalón, no un gradiente:

| Modo | OTD | Registros |
|---|---|---|
| Air | 87.21% | 15,062 |
| Parcel | 86.49% | 83,209 |
| TL | 84.79% | 41,164 |
| LTL | 84.36% | 52,089 |
| Drayage | 81.05% | 8,637 |
| **Ocean** | **66.71%** | 17,216 |

Ocean queda casi 15 puntos por debajo del siguiente modo. Dado que el dataset documenta
un episodio de congestión portuaria en Q1-2024, queda abierta la pregunta de si se trata
de una característica estructural del modo o del efecto de un evento concentrado en el
tiempo. Se aborda en el notebook 03.

## 3. Ejecución del pipeline completo

Las secciones anteriores construyeron y verificaron cada paso por separado, aplicándolos
de forma acumulativa sobre un mismo objeto. Esta sección ejecuta el pipeline completo
desde el archivo crudo, en una sola llamada, para confirmar que el resultado es
reproducible de principio a fin.

In [13]:
pipeline = LimpiadorEmbarques(df_raw)
df_limpio = pipeline.ejecutar().df

print(f"\n{'='*60}")
print(f"Entrada : {len(df_raw):,} filas × {df_raw.shape[1]} columnas")
print(f"Salida  : {len(df_limpio):,} filas × {df_limpio.shape[1]} columnas")
print(f"{'='*60}")

[1-normalizar] mode: 6,698
[1-normalizar] origin_country: 4,426
[1-normalizar] dest_country: 0
[2-deduplicar] registros eliminados: 1,320
[3-fechas] ship_date → datetime: 220,000
[3-fechas] planned_pickup_ts → datetime: 220,000
[3-fechas] planned_delivery_ts → datetime: 220,000
[3-fechas] actual_pickup_ts → datetime: 220,000
[3-fechas] actual_delivery_ts → datetime: 220,000
[4-fechas-imposibles] entregas reconstruidas: 298
[5-costos] signos invertidos: 393
[5-costos] validados contra sus componentes: 387
[5-costos] no validables por componente nulo: 6
[5-costos] corregidos que no cuadran: 0
[6-pesos] corregidos por división entre 1000: 242
[6-pesos]   Parcel: 89
[6-pesos]   LTL: 69
[6-pesos]   TL: 45
[6-pesos]   Ocean: 18
[6-pesos]   Air: 13
[6-pesos]   Drayage: 8
[7-nulos] carrier_id ausente → registro descartado: 880
[7-nulos] incoterm ausente → NO_APLICA: 6,569
[7-nulos] accessorial_usd ausente → 0: 3,290
[7-nulos] weight_kg ausente → mediana de su modo: 2,622
[7-nulos] volume_cbm a

In [14]:
print("Bitácora de ejecución\n")
print(pipeline.reporte().to_string(index=False))

Bitácora de ejecución

               paso                                       detalle  afectados
       1-normalizar                                          mode       6698
       1-normalizar                                origin_country       4426
       1-normalizar                                  dest_country          0
       2-deduplicar                          registros eliminados       1320
           3-fechas                          ship_date → datetime     220000
           3-fechas                  planned_pickup_ts → datetime     220000
           3-fechas                planned_delivery_ts → datetime     220000
           3-fechas                   actual_pickup_ts → datetime     220000
           3-fechas                 actual_delivery_ts → datetime     220000
4-fechas-imposibles                        entregas reconstruidas        298
           5-costos                             signos invertidos        393
           5-costos              validados contra sus

## 4. Validación contra la referencia oficial

Hasta aquí cada paso se verificó contra sus propios criterios internos. Esta sección
contrasta el resultado con `fact_shipments_clean.csv`, la versión limpia oficial del
dataset, que no se ha consultado en ninguna etapa anterior.

La comparación no busca coincidencia perfecta. Se conocen de antemano dos divergencias
deliberadas —el descarte de registros sin transportista y la imputación de nulos— cuyo
efecto debe poder cuantificarse y explicarse. Una coincidencia total indicaría que el
pipeline replicó la referencia en lugar de aplicar criterios propios.

In [15]:
ids_pipeline = set(df_limpio["shipment_id"])
ids_oficial = set(df_clean_ref["shipment_id"])

print("--- Cobertura de identificadores ---")
print(f"En ambos          : {len(ids_pipeline & ids_oficial):,}")
print(f"Solo en pipeline  : {len(ids_pipeline - ids_oficial):,}")
print(f"Solo en oficial   : {len(ids_oficial - ids_pipeline):,}")

--- Cobertura de identificadores ---
En ambos          : 219,120
Solo en pipeline  : 0
Solo en oficial   : 880


In [16]:
faltantes = ids_oficial - ids_pipeline
sin_carrier_raw = set(df_raw.loc[df_raw["carrier_id"].isna(), "shipment_id"])

print(f"Ausentes respecto al oficial : {len(faltantes):,}")
print(f"De ellos, sin carrier_id     : {len(faltantes & sin_carrier_raw):,}")
print(f"Ausentes por otra razón      : {len(faltantes - sin_carrier_raw):,}")

Ausentes respecto al oficial : 880
De ellos, sin carrier_id     : 880
Ausentes por otra razón      : 0


### 4.1 Cobertura de registros

| | Registros |
|---|---|
| Presentes en ambas versiones | 219,120 |
| Exclusivos del pipeline | 0 |
| Exclusivos de la referencia oficial | 880 |

Los 880 ausentes corresponden en su totalidad a registros sin `carrier_id` en el archivo
crudo, descartados por el paso 7. No hay pérdidas atribuibles a otra causa.

El cero de la fila intermedia es la comprobación relevante: el pipeline no genera
identificadores que la referencia no reconozca ni conserva registros que la limpieza
oficial haya eliminado. El resultado es un subconjunto exacto de la versión oficial,
reducido por un criterio analítico declarado.

In [17]:
# Alineamos ambas versiones por identificador
a = df_limpio.set_index("shipment_id").sort_index()
b = df_clean_ref.set_index("shipment_id").sort_index()
b = b.loc[a.index]  # solo los 219,120 compartidos

print(f"Registros comparables: {len(a):,}\n")

for col in ["mode", "origin_country", "dest_country"]:
    iguales = (a[col] == b[col]).sum()
    print(f"{col:16} {iguales:,} de {len(a):,} coinciden  ({iguales/len(a):.2%})")

Registros comparables: 219,120

mode             219,120 de 219,120 coinciden  (100.00%)
origin_country   219,120 de 219,120 coinciden  (100.00%)
dest_country     219,120 de 219,120 coinciden  (100.00%)


In [18]:
for col in ["total_cost_usd", "weight_kg", "distance_km"]:
    diff = (a[col] - b[col]).abs()
    iguales = (diff < 0.01).sum()
    print(f"{col:18} {iguales:,} de {len(a):,} coinciden  ({iguales/len(a):.2%})")
    if iguales < len(a):
        print(f"{'':18} desvío máximo: {diff.max():,.2f}")

total_cost_usd     219,120 de 219,120 coinciden  (100.00%)
weight_kg          216,502 de 219,120 coinciden  (98.81%)
                   desvío máximo: 3,996,000.00
distance_km        219,120 de 219,120 coinciden  (100.00%)


In [19]:
difiere = (a["weight_kg"] - b["weight_kg"]).abs() >= 0.01

# ¿Cuáles eran nulos en el crudo?
nulos_raw = set(df_raw.loc[df_raw["weight_kg"].isna(), "shipment_id"])
ids_difieren = set(a.index[difiere])

print(f"Registros que difieren   : {len(ids_difieren):,}")
print(f"  Eran nulos (imputados) : {len(ids_difieren & nulos_raw):,}")
print(f"  Tenían valor           : {len(ids_difieren - nulos_raw):,}")

Registros que difieren   : 2,618
  Eran nulos (imputados) : 2,616
  Tenían valor           : 2


In [20]:
ids_valor = list(ids_difieren - nulos_raw)

comp = pd.DataFrame({
    "mode": a.loc[ids_valor, "mode"],
    "pipeline": a.loc[ids_valor, "weight_kg"],
    "oficial": b.loc[ids_valor, "weight_kg"],
})
comp["factor"] = (comp["pipeline"] / comp["oficial"]).round(1)

print(comp.to_string())

               mode   pipeline  oficial  factor
shipment_id                                    
SHP1103930       TL  4000000.0   4000.0  1000.0
SHP1201298   Parcel      900.0      0.9  1000.0


### 4.2 Comparación de valores

Sobre los 219,120 registros compartidos:

| Columna | Coincidencia |
|---|---|
| `mode` | 100.00% |
| `origin_country` | 100.00% |
| `dest_country` | 100.00% |
| `total_cost_usd` | 100.00% |
| `distance_km` | 100.00% |
| `weight_kg` | 98.81% |

La coincidencia total en las categóricas valida el diccionario de sinónimos completo: cada
variante se mapeó no solo a una categoría única sino a la misma etiqueta que emplea la
versión oficial. La reducción de 14 valores a 6 podría haberse hecho eligiendo etiquetas
distintas y verse igual de limpia; el 100% confirma que también la elección fue correcta.

En `total_cost_usd`, la coincidencia total sobre los 393 registros con signo invertido
confirma la corrección del paso 5 contra una fuente externa.

**Divergencia en `weight_kg`: 2,618 registros.** El desglose separa dos causas de
naturaleza distinta:

- **2,616 son imputaciones.** Corresponden a registros que llegaron nulos al pipeline y
  recibieron la mediana de su modo. La divergencia es inherente al método: una estimación
  no reproduce un valor que no estaba en el archivo de origen.
- **2 tenían valor original.** Son SHP1103930 (TL) y SHP1201298 (Parcel), ambos con
  factor de corrupción 1000 confirmado contra la referencia.

**Sobre los 2 no corregidos.** En ambos el criterio de ida y vuelta se abstuvo por la
misma razón: el valor dividido entre 1000 cae por debajo del percentil 1 de su modo —
4,000 kg frente a un piso de 4,884.8 para TL; 0.9 kg para un Parcel cuyo p01 es superior.

Esto revela una limitación estructural del criterio, no un error de implementación: al
usar el p01 como cota inferior, el método no puede corregir valores cuyo original
pertenezca al 1% más ligero de su modo. La cobertura real es de 242 sobre 244 casos
(99.2%), y el README del dataset sobreestimaba la cifra al situarla cerca de 250.

Ampliar el rango de aceptación permitiría capturar estos dos casos, a costa de admitir
correcciones hacia valores cada vez menos plausibles. Se documenta la limitación en lugar
de ajustar el umbral hasta alcanzar la cifra esperada.

In [21]:
for col in ["actual_delivery_ts", "actual_pickup_ts", "ship_date"]:
    ambos = a[col].notna() & b[col].notna()
    iguales = (a.loc[ambos, col] == b.loc[ambos, col]).sum()
    print(f"{col:22} {iguales:,} de {ambos.sum():,} coinciden  ({iguales/ambos.sum():.2%})")

actual_delivery_ts     0 de 217,377 coinciden  (0.00%)
actual_pickup_ts       219,120 de 219,120 coinciden  (100.00%)
ship_date              219,120 de 219,120 coinciden  (100.00%)


In [22]:
print("Tipos de dato:")
print(f"  pipeline : {a['actual_delivery_ts'].dtype}")
print(f"  oficial  : {b['actual_delivery_ts'].dtype}")

print("\nPrimeros valores:")
print(f"  pipeline : {a['actual_delivery_ts'].iloc[0]!r}")
print(f"  oficial  : {b['actual_delivery_ts'].iloc[0]!r}")

Tipos de dato:
  pipeline : datetime64[us]
  oficial  : str

Primeros valores:
  pipeline : Timestamp('2025-12-02 12:18:00')
  oficial  : '2025-12-02 12:18:00.000000000'


In [23]:
b_fechas = b[COLUMNAS_FECHA].apply(pd.to_datetime).astype("datetime64[us]")

print("--- Comparación con tipos alineados ---")
for col in COLUMNAS_FECHA:
    ambos = a[col].notna() & b_fechas[col].notna()
    iguales = (a.loc[ambos, col] == b_fechas.loc[ambos, col]).sum()
    print(f"{col:22} {iguales:,} de {ambos.sum():,}  ({iguales/ambos.sum():.2%})")

--- Comparación con tipos alineados ---
ship_date              219,120 de 219,120  (100.00%)
planned_pickup_ts      219,120 de 219,120  (100.00%)
planned_delivery_ts    219,120 de 219,120  (100.00%)
actual_pickup_ts       219,120 de 219,120  (100.00%)
actual_delivery_ts     217,377 de 217,377  (100.00%)


### 4.3 Comparación de fechas

| Columna | Coincidencia |
|---|---|
| `ship_date` | 100.00% |
| `planned_pickup_ts` | 100.00% |
| `planned_delivery_ts` | 100.00% |
| `actual_pickup_ts` | 100.00% |
| `actual_delivery_ts` | 100.00% |

**Validación del paso 4.** Las 298 fechas reconstruidas coinciden con la versión oficial
al segundo. La regla derivada en el diagnóstico —`actual_pickup_ts` más
`actual_transit_days`— reproduce el criterio de la limpieza de referencia.

El resultado es más significativo de lo que sugiere la cifra. La reconstrucción se eligió
tras descartar la hipótesis del desplazamiento: sumar 48 horas a la fecha corrupta daba
un tránsito de cero días, lo que indicaba que el valor original se había borrado, no
movido. La alternativa fue derivarlo de `actual_transit_days`, columna que la corrupción
no alteró. Nada garantizaba que ese fuera el mismo camino seguido por quien generó el
dataset; la coincidencia exacta en los 298 casos lo confirma.

**Nota metodológica.** La comparación inicial arrojó 0.00% de coincidencia en
`actual_delivery_ts`. La causa no era analítica sino de tipos: `fact_shipments_clean.csv`
se carga con las fechas como texto, y comparar un `Timestamp` contra una cadena devuelve
`False` en todos los casos. Un cero absoluto es indicio de artefacto técnico antes que de
error de lógica — un fallo real habría afectado a los 298 registros reconstruidos, no a
los 217,377.

## 5. Exportación del dataset procesado

El resultado se guarda en dos formatos con destinos distintos:

- **Pickle** — conserva los tipos de dato exactos (`datetime64[us]`, `boolean` con nulos)
  y es el formato de entrada para el notebook 03. Solo lo lee Python.
- **CSV** — formato de intercambio para la carga en Power BI y para consulta desde SQL.
  Pierde la información de tipos, que deberá declararse en el destino.

Se documenta la limitación conocida del entorno: `to_parquet()` falla con `ArrowKeyError`
por un conflicto entre la versión de pyarrow instalada y Python 3.14. Parquet sería la
opción preferible por preservar tipos y comprimir mejor; se usa pickle como sustituto.

In [24]:
RUTA_PROC = Path.cwd().parent / "data" / "processed"

df_limpio.to_pickle(RUTA_PROC / "02_shipments_limpio.pkl")
df_limpio.to_csv(RUTA_PROC / "02_shipments_limpio.csv", index=False)
pipeline.reporte().to_csv(RUTA_PROC / "02_bitacora_limpieza.csv", index=False)

for archivo in sorted(RUTA_PROC.glob("02_*")):
    mb = archivo.stat().st_size / 1024**2
    print(f"{archivo.name:32} {mb:>8.1f} MB")

02_bitacora_limpieza.csv              0.0 MB
02_shipments_limpio.csv              67.9 MB
02_shipments_limpio.pkl              89.8 MB


| Archivo | Tamaño | Destino |
|---|---|---|
| `02_shipments_limpio.pkl` | 89.8 MB | Notebook 03 (Python) |
| `02_shipments_limpio.csv` | 67.9 MB | Power BI, SQL |
| `02_bitacora_limpieza.csv` | < 0.1 MB | Documentación del proceso |

El pickle resulta mayor que el CSV. Almacena cada valor en su representación binaria de
ancho fijo —ocho bytes por float de 64 bits— mientras que el texto usa solo los
caracteres necesarios: `11.1` ocupa cuatro. En un dataset con muchas columnas numéricas
de valores cortos, el formato de texto resulta más compacto. La ventaja del pickle está
en la velocidad de lectura y en la preservación de tipos, no en el tamaño.

## 6. Conclusiones

### 6.1 Resultado

El pipeline transforma `fact_shipments_raw.csv` (221,320 × 39) en un dataset analítico de
219,120 registros × 43 columnas, aplicando ocho reglas derivadas del diagnóstico de
calidad. La ejecución completa desde el archivo crudo reproduce las cifras obtenidas
durante la construcción paso a paso.

La validación contra `fact_shipments_clean.csv` arroja coincidencia total en `mode`,
`origin_country`, `dest_country`, `total_cost_usd`, `distance_km` y las cinco columnas
temporales. La única divergencia de valores está en `weight_kg` (98.81%), atribuible en
2,616 de 2,618 casos a la imputación por mediana.

### 6.2 Lo que la validación demuestra

Tres resultados confirman hipótesis del diagnóstico contra una fuente externa que no se
consultó durante la construcción:

**La corrección de signo en los costos.** Los 393 registros con `total_cost_usd` negativo
coinciden exactamente con la versión oficial. La hipótesis de que la corrupción invertía
el signo sin afectar los componentes queda verificada.

**La reconstrucción de fechas.** Las 298 entregas recalculadas como `actual_pickup_ts`
más `actual_transit_days` coinciden al segundo. La regla se eligió tras descartar la
hipótesis del desplazamiento; nada garantizaba que coincidiera con el criterio de origen.

**El diccionario de sinónimos.** La coincidencia total en las categóricas confirma que
cada variante se mapeó no solo a un valor único, sino a la misma etiqueta que emplea la
referencia.

### 6.3 Limitaciones

**Cobertura de la corrección de pesos: 242 de 244.** Los dos casos restantes —SHP1103930
(TL) y SHP1201298 (Parcel)— tienen factor de corrupción 1000 confirmado, pero el criterio
de ida y vuelta se abstiene porque el valor corregido caería por debajo del percentil 1 de
su modo. Es una limitación estructural: el método no puede corregir valores cuyo original
pertenezca al extremo ligero de la distribución. Ampliar el rango de aceptación los
capturaría a costa de admitir correcciones hacia valores cada vez menos plausibles.

**Divergencia de 880 registros.** El descarte de embarques sin `carrier_id` es una
decisión analítica, no un error. Sin transportista el registro no alimenta el scorecard,
que es el eje del proyecto. La referencia oficial los conserva.

**Imputación de `weight_kg` y `volume_cbm`.** La mediana por modo es una estimación. Los
2,616 registros imputados no reproducen el valor original y no deben tratarse como
mediciones al analizar costo por kilogramo.

**Nulos en `actual_delivery_ts`.** No es posible distinguir un embarque en tránsito de una
pérdida de dato. El OTD se calcula sobre 217,377 entregas confirmadas, con el denominador
declarado; los 1,743 restantes quedan como indeterminados y no como incumplimientos.

**Naturaleza de los datos.** El dataset es sintético. Los hallazgos demuestran una
metodología de auditoría, no condiciones operativas de una empresa real.

### 6.4 Hallazgos para la fase de análisis

El diagnóstico y la limpieza dejan cuatro líneas abiertas para el notebook 03:

| Observación | Pregunta |
|---|---|
| Ocean registra 66.71% de OTD frente a 81–87% del resto | ¿Característica del modo o efecto de la congestión portuaria de Q1-2024? |
| El retraso mediano es de −18.44 horas | ¿La planeación es sistemáticamente holgada? |
| El tier Core acumula más costo de claims que Strategic | ¿Contradice el supuesto de que el tier táctico concentra el riesgo? |
| 68 lanes de RFP no se adjudicaron al postor más barato | ¿Cuál es el sobrecosto anualizado? |

### 6.5 Siguiente etapa

Análisis exploratorio y de negocio sobre `data/processed/02_shipments_limpio.pkl`,
incorporando `fact_claims`, `fact_rfp_bids` y las tablas de dimensiones.